# 📊 Visualisation et dashboard

> ✍️ **Notebook dirigé** · complétez les cellules marquées ✍️ À VOUS.

**Jour 2 · 9h45** · Exercice

*Analyse exploratoire et interprétation métier*

**Objectifs**
- Construire des visualisations analytiques et communicantes
- Monter un dashboard interactif Plotly
- Y intégrer la représentativité des segments et des alertes sur les biais

In [21]:
%pip install -q fairlearn
# ⚙️ À exécuter en premier : récupère la configuration du cours (quelques secondes)
import os, sys
CONFIG_ID = "1bW5RsToUHc-7pGiyHS_1B2OYkAFsrv3r"

# Suppression du fichier corrompu s'il existe
if os.path.exists("config_projet.py"):
    os.remove("config_projet.py")

# Téléchargement propre avec gdown pour contourner l'avertissement de virus Google Drive
!gdown --id {CONFIG_ID} -O config_projet.py

print("✅ Configuration prête : chaque fichier de données se télécharge au premier usage")

/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1bW5RsToUHc-7pGiyHS_1B2OYkAFsrv3r
From (redirected): https://drive.google.com/uc?id=1bW5RsToUHc-7pGiyHS_1B2OYkAFsrv3r&confirm=t&uuid=52383dd5-655c-43f7-9b21-3de5394927f0
To: /content/config_projet.py
100% 6.94k/6.94k [00:00<00:00, 21.2MB/s]
✅ Configuration prête : chaque fichier de données se télécharge au premier usage


In [22]:
# Choisissez votre secteur : "commerce", "banque" ou "hotellerie"
SECTEUR = "commerce"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from config_projet import charger, reference, nettoyage_minimal

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
brut, cfg = charger(SECTEUR)
print(cfg["nom"], "·", brut.shape[0], "lignes,", brut.shape[1], "colonnes")

Maison Lumen · 24224 lignes, 16 colonnes


In [23]:
import plotly.express as px
import plotly.graph_objects as go
from config_projet import exporter_html, PALETTE
from fairlearn.metrics import MetricFrame, selection_rate, count
df = nettoyage_minimal(brut, cfg)
for col in cfg["sensibles"]:
    df[col] = df[col].fillna("inconnu")
mensuel = df.set_index(cfg["date"])[cfg["ca"]].resample("MS").sum()

## 1. Les chiffres clés
Trois indicateurs `go.Indicator` : chiffre d'affaires de 2025 avec l'évolution par rapport à 2024, part de votre segment principal, taux de décision favorable.

In [24]:
ca_annee=mensuel.groupby(mensuel.index.year).sum()
ca24=float(ca_annee.loc[2024]); ca25=float(ca_annee.loc[2025])
par_canal=df.groupby(cfg["segment"])[cfg["ca"]].sum().sort_values(ascending=False)
principal=par_canal.index[0]; part=par_canal.iloc[0]/par_canal.sum()*100
taux_global=df[cfg["decision"]].mean()*100
kpis=go.Figure()
kpis.add_trace(go.Indicator(mode="number+delta",value=ca25,number={"suffix":" €","valueformat":",.0f"},delta={"reference":ca24,"relative":True,"valueformat":".1%"},title={"text":"CA 2025 · évolution / 2024"},domain={"x":[0,.32],"y":[0,1]}))
kpis.add_trace(go.Indicator(mode="number",value=part,number={"suffix":" %","valueformat":".1f"},title={"text":f"Part du CA · {principal}"},domain={"x":[.34,.65],"y":[0,1]}))
kpis.add_trace(go.Indicator(mode="number",value=taux_global,number={"suffix":" %","valueformat":".1f"},title={"text":"Commandes précédées d’une offre"},domain={"x":[.68,1],"y":[0,1]}))
kpis.update_layout(height=240,template="plotly_white")
kpis.show()
print(f"Le CA 2025 atteint {ca25:,.0f} €, soit {(ca25/ca24-1)*100:+.1f} % sur un an. Le canal {principal} représente {part:.1f} % des ventes de toute la période.")

Le CA 2025 atteint 702,326 €, soit +6.7 % sur un an. Le canal magasin représente 49.4 % des ventes de toute la période.


## 2. La tendance, pour les décideurs
Courbe mensuelle, tendance sur 12 mois, et l'événement du brief annoté. Le titre dit la conclusion.

In [25]:
# ✍️ À VOUS : remplacez les ... et traitez les TODO
courbe = px.line(mensuel.rename("CA").reset_index(), x=cfg["date"], y="CA",
                 title="Le chiffre d'affaires suit un cycle annuel marqué", color_discrete_sequence=PALETTE)
courbe.add_scatter(x=mensuel.index, y=mensuel.rolling(12, center=True).mean(), name="tendance 12 mois",
                   line=dict(width=4, color=PALETTE[1]))
courbe.update_layout(yaxis_title="€", xaxis_title=""); courbe

## 3. L'analyse, pour l'équipe data
Un graphique plus riche : chiffre d'affaires mensuel par segment, avec un sélecteur interactif.

In [26]:
# Aggregation du chiffre d'affaires par mois et par segment
segment_col = cfg["segment"]
ca_col = cfg["ca"]
date_col = cfg["date"]

# Groupement par mois (début de mois 'MS') et par segment
df_segment = df.groupby([pd.Grouper(key=date_col, freq="MS"), segment_col])[ca_col].sum().reset_index()

# Liste des segments uniques
segments = df_segment[segment_col].unique()

# Construction du graphique interactif avec graph_objects
fig_segment = go.Figure()

# Ajout d'une courbe pour chaque segment
for i, seg in enumerate(segments):
    df_seg = df_segment[df_segment[segment_col] == seg]
    fig_segment.add_trace(
        go.Scatter(
            x=df_seg[date_col],
            y=df_seg[ca_col],
            name=seg,
            line=dict(color=PALETTE[i % len(PALETTE)], width=3),
            visible=True
        )
    )

# Configuration des boutons d'interaction (sélecteur)
buttons = []

# Option "Tous les segments"
buttons.append(dict(
    method="update",
    label="Tous les segments",
    args=[{"visible": [True] * len(segments)},
          {"title": "Chiffre d'affaires mensuel - Tous les segments"}]
))

# Une option par segment individuel
for i, seg in enumerate(segments):
    visibility = [False] * len(segments)
    visibility[i] = True
    buttons.append(dict(
        method="update",
        label=f"Segment : {seg}",
        args=[{"visible": visibility},
              {"title": f"Chiffre d'affaires mensuel - Segment {seg}"}]
    ))

# Mise à jour du layout avec le sélecteur interactif
fig_segment.update_layout(
    updatemenus=[
        dict(
            active=0,
            buttons=buttons,
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.0,
            xanchor="left",
            y=1.15,
            yanchor="top"
        )
    ],
    title="Chiffre d'affaires mensuel par segment",
    xaxis_title="",
    yaxis_title="Chiffre d'affaires (€)",
    hovermode="x unified",
    height=500
)

fig_segment.show()

## 4. L'indicateur de représentativité
Pour chaque variable sensible, le ratio part observée ÷ part de référence, coloré selon le statut, avec les seuils 0,8 et 1,25.

In [27]:
clients = df.drop_duplicates("client_id")
ref = reference().query("variable == 'tranche_age'").set_index("modalite")["part_reference"]
ratio = (clients["tranche_age"].value_counts(normalize=True) / ref).round(2).rename("ratio").rename_axis("modalite").reset_index()
ratio["statut"] = np.select([ratio["ratio"] < .8, ratio["ratio"] > 1.25], ["sous-représenté", "sur-représenté"], "conforme")
rep_age = px.bar(ratio, x="modalite", y="ratio", color="statut", text="ratio",
             color_discrete_map={"sous-représenté": "#D55E00", "conforme": "#999999", "sur-représenté": "#0072B2"},
             title="Les plus de 65 ans sont sous-représentés parmi les clients fidélité")
rep_age.add_hline(y=0.8, line_dash="dash"); rep_age.add_hline(y=1.25, line_dash="dash")
rep_age.update_layout(xaxis_title="tranche d'âge", yaxis_title="part observée ÷ part de référence"); rep_age

In [28]:
clients = df.drop_duplicates("client_id")
ref = reference().query("variable == 'genre'").set_index("modalite")["part_reference"]
ratio = (clients["genre"].value_counts(normalize=True) / ref).round(2).rename("ratio").rename_axis("modalite").reset_index()
ratio["statut"] = np.select([ratio["ratio"] < .8, ratio["ratio"] > 1.25], ["sous-représenté", "sur-représenté"], "conforme")
rep_genre = px.bar(ratio, x="modalite", y="ratio", color="statut", text="ratio",
             color_discrete_map={"sous-représenté": "#D55E00", "conforme": "#999999", "sur-représenté": "#0072B2"},
             title="Les hommes sont sous-représenter")
rep_genre.add_hline(y=0.8, line_dash="dash"); rep_genre.add_hline(y=1.25, line_dash="dash")
rep_genre.update_layout(xaxis_title="Genre", yaxis_title="part observée ÷ part de référence"); rep_genre

In [29]:
df = df[df['region'].notna()]
df.isna().sum()

,0
date_commande,0
commande_id,0
client_id,0
genre,0
age,0
tranche_age,0
region,0
canal,0
categorie,0
montant_eur,0


In [30]:
clients = df.drop_duplicates("client_id")
# Filtrer pour exclure les valeurs 'inconnu' de la région
clients_region = clients[clients["region"] != "inconnu"]

ref = reference().query("variable == 'region'").set_index("modalite")["part_reference"]
ratio = (clients_region["region"].value_counts(normalize=True) / ref).round(2).rename("ratio").rename_axis("modalite").reset_index()
ratio["statut"] = np.select([ratio["ratio"] < .8, ratio["ratio"] > 1.25], ["sous-représenté", "sur-représenté"], "conforme")

rep_region = px.bar(ratio, x="modalite", y="ratio", color="statut", text="ratio",
             color_discrete_map={"sous-représenté": "#D55E00", "conforme": "#999999", "sur-représenté": "#0072B2"},
             title="L'Ile de France est sur-représentée tandis que le Nord-Est et l'Outre-mer sont sous-représentés")
rep_region.add_hline(y=0.8, line_dash="dash"); rep_region.add_hline(y=1.25, line_dash="dash")
rep_region.update_layout(xaxis_title="Région", yaxis_title="part observée ÷ part de référence"); rep_region

## 5. La parité de la décision
Taux de décision favorable par groupe, avec la ligne des quatre cinquièmes du meilleur taux.

In [31]:
def calculer_taux_decision_favorable(colonne_sensible):
    # Calcul du taux de décision favorable par groupe
    decision_col = cfg["decision"]

    # Filtrer les 'inconnu' pour l'analyse d'équité si nécessaire
    df_clean = df[df[colonne_sensible] != "inconnu"]

    # Calcul du taux de sélection (décision favorable) par groupe
    metrics = MetricFrame(
        metrics=selection_rate,
        y_true=df_clean[decision_col],
        y_pred=df_clean[decision_col],  # Dans notre cas, y_pred représente la décision prise
        sensitive_features=df_clean[colonne_sensible]
    )

    rates = metrics.by_group.reset_index()
    rates.columns = [colonne_sensible, "taux_selection"]
    return rates

# Exemple de calcul pour la tranche d'âge
taux_age = calculer_taux_decision_favorable("tranche_age")
display(taux_age)

,tranche_age,taux_selection
0,18-24,0.477915
1,25-34,0.483785
2,35-44,0.384951
3,45-54,0.401703
4,55-64,0.247549
5,65+,0.173943


In [32]:
# Calcul du taux pour la tranche d'âge
taux_age = calculer_taux_decision_favorable("tranche_age")

# Identification du meilleur taux de sélection pour tracer le seuil des 4/5
meilleur_taux = taux_age["taux_selection"].max()
seuil_4_cinquieme = meilleur_taux * 0.8

# Attribution du statut par rapport au seuil des 4/5
taux_age["statut"] = np.where(taux_age["taux_selection"] < seuil_4_cinquieme, "discrimination suspectée", "conforme")

# Tracé du graphique avec Plotly Express
parite = px.bar(
    taux_age,
    x="tranche_age",
    y="taux_selection",
    color="statut",
    text=taux_age["taux_selection"].apply(lambda x: f"{x:.1%}"),
    color_discrete_map={"discrimination suspectée": "#D55E00", "conforme": "#009E73"},
    title="Taux de décision favorable par tranche d'âge (seuil des 4/5 du meilleur groupe)"
)

# Ajout de la ligne représentant le seuil des 4/5
parite.add_hline(
    y=seuil_4_cinquieme,
    line_dash="dash",
    line_color="#000000",
    annotation_text=f"Seuil 4/5 ({seuil_4_cinquieme:.1%})",
    annotation_position="top left"
)

# Mise à jour des axes
parite.update_layout(
    xaxis_title="Tranche d'âge",
    yaxis_title="Taux de sélection",
    yaxis_tickformat=".0%"
)

parite

### Parité de décision par Genre

In [33]:
# Calcul du taux pour le genre
taux_genre = calculer_taux_decision_favorable("genre")

# Meilleur taux et seuil des 4/5
meilleur_taux_genre = taux_genre["taux_selection"].max()
seuil_genre = meilleur_taux_genre * 0.8

# Attribution du statut
taux_genre["statut"] = np.where(taux_genre["taux_selection"] < seuil_genre, "discrimination suspectée", "conforme")

# Tracé
parite_genre = px.bar(
    taux_genre,
    x="genre",
    y="taux_selection",
    color="statut",
    text=taux_genre["taux_selection"].apply(lambda x: f"{x:.1%}"),
    color_discrete_map={"discrimination suspectée": "#D55E00", "conforme": "#009E73"},
    title="Taux de décision favorable par genre (seuil des 4/5)"
)

parite_genre.add_hline(
    y=seuil_genre,
    line_dash="dash",
    line_color="#000000",
    annotation_text=f"Seuil 4/5 ({seuil_genre:.1%})",
    annotation_position="top left"
)

parite_genre.update_layout(
    xaxis_title="Genre",
    yaxis_title="Taux de sélection",
    yaxis_tickformat=".0%"
)

parite_genre

### Parité de décision par Région

In [34]:
# Calcul du taux pour la région
taux_region = calculer_taux_decision_favorable("region")

# Meilleur taux et seuil des 4/5
meilleur_taux_region = taux_region["taux_selection"].max()
seuil_region = meilleur_taux_region * 0.8

# Attribution du statut
taux_region["statut"] = np.where(taux_region["taux_selection"] < seuil_region, "discrimination suspectée", "conforme")

# Tracé
parite_region = px.bar(
    taux_region,
    x="region",
    y="taux_selection",
    color="statut",
    text=taux_region["taux_selection"].apply(lambda x: f"{x:.1%}"),
    color_discrete_map={"discrimination suspectée": "#D55E00", "conforme": "#009E73"},
    title="Taux de décision favorable par région (seuil des 4/5)"
)

parite_region.add_hline(
    y=seuil_region,
    line_dash="dash",
    line_color="#000000",
    annotation_text=f"Seuil 4/5 ({seuil_region:.1%})",
    annotation_position="top left"
)

parite_region.update_layout(
    xaxis_title="Région",
    yaxis_title="Taux de sélection",
    yaxis_tickformat=".0%"
)

parite_region

## 6. Les alertes
Un bloc HTML qui liste en clair les groupes sous-représentés et les groupes sous la ligne des 4/5. Ne comptez pas sur la couleur seule : écrivez-le.

In [35]:
# Définition textuelle des alertes détectées dans nos analyses
alertes = """
<div style='background-color: #F8D7DA; color: #721C24; padding: 15px; border-radius: 5px; margin-bottom: 20px;'>
    <h3>⚠️ Alertes de représentativité et d'équité</h3>
    <ul>
        <li><b>Sous-représentation démographique :</b> Les plus de 55 ans et les hommes sont sous-représentés par rapport à la population de référence. Sur le plan géographique, le Nord-Est et l'Outre-mer sont également sous-représentés.</li>
        <li><b>Sur-représentation démographique :</b> Ceux entre 18 et 54 ans sont sur-représentés par rapport à la population de référence. Sur le plan géographique, l'île-de-France est également sur-représentés.</li>
        <li><b>Biais de décision (Règle des 4/5) :</b> Les seniors (tranches d'âge 55-64 ans et 65+) présentent des taux de décision favorables sous le seuil d'équité par rapport à la tranche d'âge de référence (25-34 ans).</li>
        <li><b>Biais de décision (Règle des 4/5) :</b> Ceux en outre-mer présentent des taux de décision favorables sous le seuil d'équité par rapport à la région de référence île-de-France.</li>
    </ul>
</div>
"""
from IPython.display import HTML
HTML(alertes)

## 7. L'export HTML autonome
Assemblez le tout avec `exporter_html` (fourni) : une page pour les décideurs, une annexe pour l'équipe data. Chaque figure reçoit une description courte, utile aux lecteurs d'écran.

In [36]:
# Exportation des éléments du tableau de bord dans un livrable HTML mis à jour
contexte="""<section><h2>Périmètre et lecture</h2><p>Données fictives : commandes de janvier 2023 à décembre 2025. Doublons exacts et avoirs négatifs exclus conformément au cours. Le total est un CA TTC des ventes retenues, pas un CA comptable net des avoirs. Les remises sont déjà déduites des montants.</p><p>Une offre est un envoi dans le mois précédent la commande. Le fichier n’observe pas tous les clients sans achat. La représentativité compte les clients uniques ; la parité compte les commandes.</p></section>"""
recommandations="""<section><h2>Ce que je proposerais à la direction</h2><p>Suivre le CA et le panier moyen par canal, en comparant les mêmes mois d’une année à l’autre. Vérifier les règles de ciblage des offres pour les clients âgés, puis comparer des profils proches à canal égal. Compléter ces tableaux par le taux de retour, la satisfaction et les écarts d’accès aux offres.</p><p>La refonte est un repère temporel, pas une preuve de causalité. Le fichier ne contient pas d’émissions ou de données de livraison permettant de calculer un indicateur environnemental fiable.</p></section>"""
blocs_dashboard = [
    contexte,
    ("Chiffres clés", kpis, "Chiffre d'affaires de l'année en cours et progression relative par rapport à l'année précédente."),
    ("Tendance des ventes", courbe, "Évolution temporelle du chiffre d'affaires et courbe de tendance sur un glissement de 12 mois."),
    ("Analyse par canal", fig_segment, "Chiffre d'affaires mensuel ventilé par canal de vente avec sélecteur interactif."),
    ("Représentativité par Âge", rep_age, "Comparaison de la structure par âge de la clientèle par rapport à la population de référence."),
    ("Représentativité par Genre", rep_genre, "Comparaison de la répartition par genre de la clientèle par rapport à la population de référence."),
    ("Représentativité par Région", rep_region, "Comparaison de la répartition régionale de la clientèle par rapport à la population de référence."),
    ("Biais par Tranche d'âge", parite, "Vérification de la règle d'équité des 4/5èmes sur le taux d'attribution par tranche d'âge."),
    ("Biais par Genre", parite_genre, "Vérification de la règle d'équité des 4/5èmes par genre."),
    ("Biais par Région", parite_region, "Vérification de la règle d'équité des 4/5èmes par région."),
    alertes,
    recommandations
]

taille = exporter_html(f"{cfg['nom']} · Tableau de bord complet", blocs_dashboard, f"dashboard_{SECTEUR}.html")
print(f"dashboard_{SECTEUR}.html : {taille:.1f} Mo")
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(f"dashboard_{SECTEUR}.html")

dashboard_commerce.html : 4.7 Mo


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## ✅ Check-list du livrable
- [ ] Le fichier s'ouvre seul dans un navigateur, sans connexion.
- [ ] Chaque graphique a un titre qui conclut et une description.
- [ ] Les alertes sont écrites, pas seulement colorées.
- [ ] Les couleurs restent lisibles pour un daltonien (palette Okabe-Ito).

💾 Enregistrez le notebook dans GitHub (« Dashboard Plotly exporté ») et envoyez le fichier HTML téléchargé au formateur.